# 02: Multi-Phase Binding Kinetics Modeling

## Overview
This notebook simulates the molecular capture efficiency at the LFA test line. It models the competition between high-affinity Nanobodies (Nbs) and Monoclonal Antibodies (mAbs) for target capture across three distinct operational phases.

### The 3-Phase Model:
1. Phase 1 (Arrival): Target-nanoparticle complex reaches the test line (t=0 to 2 min).
2. Phase 2 (Binding): Primary binding interaction between complex and printed antibodies.
3. Phase 3 (Wash/Buffer Flow): Purification and dissociation phase before signal amplification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. KINETIC PARAMETERS (Comparison: Nb vs mAb)
complexes = {
    "Nanobody_59H10": {"ka": 1.2e5, "kd": 5.4e-4, "Rmax": 100},
    "mAb_CPHIV-1": {"ka": 0.8e5, "kd": 2.1e-3, "Rmax": 100}
}

times = np.linspace(0, 600, 1000)      # 10 minute total window
c_target = 10e-9                       # 10 nM concentration

# 2. BINDING SIMULATION (Langmuir Kinetic Model)
def simulate_binding(params, time_array, conc):
    ka, kd, Rmax = params['ka'], params['kd'], params['Rmax']
    req = (ka * conc * Rmax) / (ka * conc + kd)
    ktot = ka * conc + kd
    R_assoc = req * (1 - np.exp(-ktot * time_array))
    return R_assoc

# 3. ANALYSIS & VISUALIZATION
plt.figure(figsize=(8, 5))
for name, p in complexes.items():
    binding_curve = simulate_binding(p, times, c_target)
    plt.plot(times, binding_curve, label=name)
    
    # Highlight binding level at 300 s (End of Phase 2)
    level_300 = np.interp(300, times, binding_curve)
    plt.scatter(300, level_300)
    print(f"{name} Binding Level at 5 min: {level_300:.2f}%")

plt.axvline(x=120, color='gray', linestyle='--', label='Phase 1 End')
plt.axvline(x=300, color='red', linestyle='--', label='Phase 2 End (Wash Start)')
plt.title("Comparative Binding Kinetics: Automated LFIA Detection")
plt.xlabel("Time (s)")
plt.ylabel("Relative Binding Level (%)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()